# PCA: метод главных компонент

PCA (Principal Component Analysis) уменьшает число признаков, стараясь сохранить как можно больше изменчивости данных. Новые признаки называются **главными компонентами**.

## Содержание
1. [Зачем нужен PCA](#1-зачем-нужен-pca)
2. [Геометрическая идея](#2-геометрическая-идея)
3. [Центрирование и масштабирование](#3-центрирование-и-масштабирование)
4. [Ковариационная матрица и собственные векторы](#4-ковариационная-матрица-и-собственные-векторы)
5. [Связь PCA с SVD](#5-связь-pca-с-svd)
6. [PCA с нуля через SVD](#6-pca-с-нуля-через-svd)
7. [PCA в scikit-learn](#7-pca-в-scikit-learn)
8. [Как выбрать число компонент](#8-как-выбрать-число-компонент)
9. [Визуализация Iris](#9-визуализация-iris)
10. [PCA внутри ML-пайплайна](#10-pca-внутри-ml-пайплайна)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## 1. Зачем нужен PCA

Во многих таблицах признаки связаны друг с другом. Например, площадь комнат и общая площадь дома несут похожую информацию. PCA заменяет исходные признаки меньшим числом новых признаков.

PCA используют, чтобы:

- визуализировать многомерные данные в 2D или 3D;
- сжимать данные;
- уменьшать мультиколлинеарность;
- ускорять некоторые модели;
- удалять направления с очень маленькой изменчивостью.

PCA — метод **без учителя**: при поиске компонент он использует только матрицу признаков $X$ и не смотрит на целевую переменную $y$. Поэтому направление с большой дисперсией не обязательно является самым полезным для предсказания target.

## 2. Геометрическая идея

Представим вытянутое облако точек. Первая главная компонента направлена вдоль самого длинного направления облака. Проекция на эту ось сохраняет максимальную дисперсию.

Вторая компонента:

- перпендикулярна первой;
- сохраняет максимум оставшейся дисперсии.

Следующие компоненты строятся так же. Все компоненты ортогональны друг другу. Если данные почти лежат вдоль одной линии, одной компоненты может быть достаточно вместо двух исходных признаков.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
base = rng.normal(size=250)
X_demo = np.column_stack([
    2.5 * base + rng.normal(0, 0.35, len(base)),
    1.2 * base + rng.normal(0, 0.25, len(base)),
])

pca_demo = PCA(n_components=2).fit(X_demo)
center = pca_demo.mean_

plt.figure(figsize=(7, 5))
plt.scatter(X_demo[:, 0], X_demo[:, 1], s=18, alpha=0.45)
colors = ['crimson', 'darkgreen']
for component, variance, color in zip(
    pca_demo.components_, pca_demo.explained_variance_, colors
):
    end = center + component * 2 * np.sqrt(variance)
    plt.annotate('', xy=end, xytext=center,
                 arrowprops=dict(color=color, width=2, headwidth=9))
plt.title('Главные компоненты направлены вдоль разброса данных')
plt.axis('equal')
plt.show()

## 3. Центрирование и масштабирование

### Центрирование

Сначала из каждого признака вычитают его среднее:

$$X_c=X-\bar X.
$$

После этого центр данных находится в нуле. `sklearn.PCA` выполняет центрирование автоматически.

### Масштабирование

PCA ищет направления с максимальной дисперсией. Признак, измеряемый тысячами, может доминировать над признаком со значениями от 0 до 1 только из-за единиц измерения. Поэтому признаки с разными шкалами часто стандартизируют через `StandardScaler` **до** PCA.

Масштабирование не всегда обязательно. Если все признаки измерены в сопоставимых единицах и их абсолютная дисперсия содержательно важна, можно оставить исходный масштаб. Это решение зависит от смысла данных.

In [ ]:
X_scale = np.column_stack([rng.normal(0, 1, 300), rng.normal(0, 100, 300)])
raw_pca = PCA().fit(X_scale)
scaled_pca = PCA().fit(StandardScaler().fit_transform(X_scale))

print('Доля дисперсии без масштабирования:', np.round(raw_pca.explained_variance_ratio_, 4))
print('Доля дисперсии после масштабирования:', np.round(scaled_pca.explained_variance_ratio_, 4))

## 4. Ковариационная матрица и собственные векторы

Для центрированной матрицы $X_c$ ковариационная матрица равна

$$C=\frac{1}{n-1}X_c^T X_c.
$$

Она показывает дисперсии признаков и их совместное изменение. Для неё решается задача

$$Cv_j=\lambda_jv_j.
$$

Здесь:

- собственный вектор $v_j$ задаёт направление главной компоненты;
- собственное значение $\lambda_j$ показывает дисперсию вдоль этого направления.

Компоненты сортируют по убыванию $\lambda_j$. Новые координаты первых $m$ компонент получают проекцией:

$$Z=X_cV_m.
$$

## 5. Связь PCA с SVD

SVD (Singular Value Decomposition, сингулярное разложение) представляет центрированную матрицу данных как

$$X_c=U\Sigma V^T.
$$

Матрицы имеют простой смысл:

- столбцы $V$ — направления в пространстве исходных признаков;
- диагональные элементы $\Sigma$ — сингулярные значения $\sigma_j$;
- $U\Sigma$ — координаты объектов в пространстве компонент.

Подставим SVD в ковариационную матрицу:

$$C=\frac{1}{n-1}X_c^TX_c
=\frac{1}{n-1}V\Sigma^TU^TU\Sigma V^T
=V\frac{\Sigma^2}{n-1}V^T.
$$

Поскольку $U^TU=I$, правые сингулярные векторы $V$ являются собственными векторами ковариационной матрицы. А собственные значения связаны с сингулярными значениями:

$$\lambda_j=\frac{\sigma_j^2}{n-1}.
$$

Итак, PCA можно получить двумя равносильными способами:

1. Найти собственные векторы $X_c^TX_c$.
2. Выполнить SVD самой матрицы $X_c$.

На практике SVD часто удобнее и численно устойчивее: не нужно явно вычислять $X_c^T X_c$, где могут усиливаться численные ошибки. Именно поэтому реализации PCA обычно опираются на SVD или его приближённые варианты.

In [ ]:
X_centered = X_demo - X_demo.mean(axis=0)
_, singular_values, Vt = np.linalg.svd(X_centered, full_matrices=False)
covariance = np.cov(X_centered, rowvar=False)
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

print('lambda из covariance:', np.round(eigenvalues, 6))
print('sigma² / (n-1):      ', np.round(singular_values**2 / (len(X_demo) - 1), 6))
print('Модули направлений совпадают:',
      np.allclose(np.abs(Vt.T), np.abs(eigenvectors)))

Знак компоненты может отличаться: $v$ и $-v$ задают одну и ту же ось. Поэтому при сравнении направлений выше используются абсолютные значения. Это не меняет долю объяснённой дисперсии или расстояния между объектами.

## 6. PCA с нуля через SVD

Минимальная реализация состоит из центрирования, SVD и выбора первых строк $V^T$. Метод `transform` проецирует новые данные с использованием среднего и компонент, найденных только на train.

In [ ]:
class PCAFromScratch:
    def __init__(self, n_components):
        self.n_components = n_components

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        X_centered = X - self.mean_
        _, singular_values, Vt = np.linalg.svd(X_centered, full_matrices=False)
        self.components_ = Vt[:self.n_components]
        explained = singular_values**2 / (len(X) - 1)
        self.explained_variance_ = explained[:self.n_components]
        self.explained_variance_ratio_ = (explained / explained.sum())[:self.n_components]
        return self

    def transform(self, X):
        return (np.asarray(X) - self.mean_) @ self.components_.T

    def inverse_transform(self, Z):
        return np.asarray(Z) @ self.components_ + self.mean_

    def fit_transform(self, X):
        return self.fit(X).transform(X)


pca_scratch = PCAFromScratch(n_components=1)
Z_scratch = pca_scratch.fit_transform(X_demo)
X_reconstructed = pca_scratch.inverse_transform(Z_scratch)
print('Исходная форма:', X_demo.shape)
print('После PCA:', Z_scratch.shape)
print('Объяснённая дисперсия:', pca_scratch.explained_variance_ratio_.round(4))

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_demo[:, 0], X_demo[:, 1], s=18, alpha=0.35, label='Исходные точки')
plt.scatter(X_reconstructed[:, 0], X_reconstructed[:, 1], s=18, alpha=0.7, label='После 1 компоненты')
plt.axis('equal')
plt.legend()
plt.title('Восстановление: точки проецируются на главную ось')
plt.show()

## 7. PCA в scikit-learn

Основные параметры и атрибуты `sklearn.decomposition.PCA`:

| Имя | Смысл |
|---|---|
| `n_components` | сколько компонент оставить |
| `svd_solver` | способ вычисления SVD; `auto` выбирает по форме данных |
| `whiten=True` | дополнительно масштабирует компоненты до единичной дисперсии |
| `components_` | направления компонент, строки матрицы $V^T$ |
| `explained_variance_` | дисперсия каждой сохранённой компоненты |
| `explained_variance_ratio_` | доля общей дисперсии |
| `singular_values_` | соответствующие сингулярные значения |

`fit_transform(X)` обучает PCA и сразу возвращает новые координаты. Для новых объектов используют только `transform(X_new)`, не выполняя повторный `fit`.

In [ ]:
sk_pca = PCA(n_components=1).fit(X_demo)
Z_sklearn = sk_pca.transform(X_demo)

print('Доля дисперсии sklearn:', sk_pca.explained_variance_ratio_.round(4))
print('Доля дисперсии с нуля:', pca_scratch.explained_variance_ratio_.round(4))
print('Проекции совпадают с точностью до знака:',
      np.allclose(np.abs(Z_sklearn), np.abs(Z_scratch)))

## 8. Как выбрать число компонент

`explained_variance_ratio_` показывает долю общей дисперсии, сохранённую каждой компонентой. Накопленная сумма отвечает на вопрос: сколько информации в смысле дисперсии сохраняют первые $m$ компонент?

Есть два распространённых способа выбора:

1. Задать число явно: `PCA(n_components=2)`.
2. Задать желаемую долю дисперсии: `PCA(n_components=0.95)` оставит минимальное число компонент, объясняющих не менее 95%.

Порог 95% — ориентир, а не универсальное правило. Если PCA используется перед моделью, число компонент лучше проверять по итоговой CV-метрике модели.

In [ ]:
iris = load_iris()
X_iris_scaled = StandardScaler().fit_transform(iris.data)
pca_all = PCA().fit(X_iris_scaled)
cumulative = np.cumsum(pca_all.explained_variance_ratio_)

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(cumulative) + 1), cumulative, marker='o')
plt.axhline(0.95, color='red', linestyle='--', label='95%')
plt.xticks(range(1, len(cumulative) + 1))
plt.xlabel('Число компонент')
plt.ylabel('Накопленная доля дисперсии')
plt.ylim(0, 1.05)
plt.legend()
plt.title('Сколько дисперсии сохраняют компоненты Iris')
plt.show()

print('Доли по компонентам:', pca_all.explained_variance_ratio_.round(3))
print('Накопленные доли:', cumulative.round(3))

## 9. Визуализация Iris

В Iris четыре исходных признака. PCA позволяет показать каждый объект двумя координатами. Цвет ниже используется только для визуализации: при построении компонент PCA не видел названия видов.

In [ ]:
pca_2d = PCA(n_components=2)
X_iris_2d = pca_2d.fit_transform(X_iris_scaled)

plt.figure(figsize=(8, 5))
for class_id, class_name in enumerate(iris.target_names):
    mask = iris.target == class_id
    plt.scatter(X_iris_2d[mask, 0], X_iris_2d[mask, 1], s=28, label=class_name)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
plt.title('Iris в пространстве двух главных компонент')
plt.legend()
plt.show()

## 10. PCA внутри ML-пайплайна

Чтобы избежать утечки данных, `StandardScaler` и PCA должны обучаться отдельно внутри каждого CV-фолда. `Pipeline` делает это автоматически:

```text
StandardScaler -> PCA -> модель
```

PCA не гарантирует улучшения качества. На Iris всего четыре информативных признака, поэтому удаление компонент может даже немного ухудшить классификацию. Правильный вывод получают сравнением моделей на одинаковой кросс-валидации.

In [ ]:
without_pca = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)
with_pca = make_pipeline(
    StandardScaler(),
    PCA(n_components=2),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)

score_without = cross_val_score(without_pca, iris.data, iris.target, cv=5).mean()
score_with = cross_val_score(with_pca, iris.data, iris.target, cv=5).mean()
print(f'CV accuracy без PCA: {score_without:.3f}')
print(f'CV accuracy с 2 компонентами: {score_with:.3f}')